# Build the RAG-in-Goa search index on a GPU

Embedding the corpus is the only part of this project that needs a GPU, and it needs one badly: measured on the 8 GB M2 Air we develop on, encoding runs at ~20 chunks/s, so the full corpus takes over two hours and makes the laptop unusable. A free Colab T4 does the same work in a few minutes.

Everything else — retrieval, the harness, the guardrails, the API — runs on CPU on a laptop or a small Fly.io machine. So this notebook does one job and hands back artifacts.

**Runtime → Change runtime type → T4 GPU** before running anything.

| step | what it produces | roughly |
|---|---|---|
| 1–2 | GPU confirmed, repo and deps installed | 2 min |
| 3 | 4 validation shards from Hugging Face (~1.9 GB) | 3 min |
| 4 | `data/slim/*.parquet`, small row groups so reads are cheap | 6 min |
| 5 | `data/corpus/*.parquet` — 97,941 pseudo-documents | under 1 min |
| 6 | `data/index/` — ~210k chunks, HNSW + BM25 | 8–12 min |
| 7 | real queries retrieved, to prove the index works | under 1 min |
| 8–9 | one ~600 MB archive, out to Drive or Hugging Face | 5 min |

Total: roughly half an hour, most of it moving bytes rather than computing.

The shipped chunker is script-aware recursive at 1800 chars with 200 of overlap, which gives about 2.15 chunks per document — so ~210k vectors and 323 MB of raw fp32. That default lives in one place, `ragoa/chunking/registry.py`, so the benchmarked configuration and the served one cannot drift apart.

In [ ]:
#@title 1. Confirm the GPU is actually attached
# Worth being blunt about: with no GPU this notebook still runs, just 30x slower,
# and the failure is silent. Better to stop here than to discover it in step 6.
import subprocess

import torch

print(f"torch {torch.__version__}, cuda available: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU. Runtime -> Change runtime type -> T4 GPU, then run this cell again."
    )

print(f"device      : {torch.cuda.get_device_name(0)}")
free, total = torch.cuda.mem_get_info()
print(f"gpu memory  : {free / 1e9:.1f} GB free of {total / 1e9:.1f} GB")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip())

In [ ]:
#@title 2. Clone the repo and install only what the build needs
REPO = "https://github.com/redwing-381/rag-in-goa.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}

import os
import sys

if not os.path.exists("/content/rag-in-goa"):
    !git clone --depth 1 --branch {BRANCH} {REPO} /content/rag-in-goa
os.chdir("/content/rag-in-goa")
!git log --oneline -1

# Colab already ships a CUDA-matched torch. Installing our own dependency set
# unpinned would happily replace it with a CPU wheel, so torch is left alone and
# the package goes in with --no-deps.
!pip install -q pydantic pydantic-settings pyarrow bm25s PyStemmer hnswlib orjson tqdm
!pip install -q sentence-transformers
!pip install -q -e . --no-deps

if "/content/rag-in-goa" not in sys.path:
    sys.path.insert(0, "/content/rag-in-goa")

import torch

print(f"\ntorch still on cuda: {torch.cuda.is_available()}")
from ragoa.config import settings

print(f"ragoa importable, embed model: {settings.embed_model}")

In [ ]:
#@title 3. Download the four validation shards
# Only validation is used. The full dataset is 55 GB across 14 languages; the four
# validation shards we need are 1.9 GB, and the README documents that choice.
from pathlib import Path

from huggingface_hub import hf_hub_download

DATASET = "ai4bharat/MSMARCO-XI"
# Shard prefixes are 3-letter and do not always match the ISO code.
SHARDS = {"hi": "hinval", "bn": "benval", "ta": "tamval", "mr": "marval"}

raw = Path("data/raw")
raw.mkdir(parents=True, exist_ok=True)

for lang, shard in SHARDS.items():
    target = raw / f"{shard}.parquet"
    if target.exists():
        print(f"{lang}: already have {target.name} "
              f"({target.stat().st_size / 1e9:.2f} GB)")
        continue
    path = hf_hub_download(DATASET, f"validation/{shard}.parquet",
                           repo_type="dataset")
    # Symlink rather than copy: the HF cache and /content share a disk, and the
    # Colab image does not have 4 GB to spare twice over.
    target.symlink_to(path)
    print(f"{lang}: {target.name} {target.stat().st_size / 1e9:.2f} GB")

!df -h /content | tail -1

In [ ]:
#@title 4. Convert to slim parquet
# Each raw shard is a single 1.16 GB row group, so any read decodes the whole file.
# This pays that cost once and rewrites with 2,000-row groups. It also prints the
# corpus statistics that determine index size.
!python scripts/prepare_data.py --all

In [ ]:
#@title 5. Build pseudo-documents and the multilingual query set
# MS MARCO ships passages pre-chunked at ~60 words, which would make chunking a
# no-op: any strategy would just recover the passages it was handed. Concatenating
# each query's candidates into one pseudo-document, with the character span of every
# gold passage recorded, is what makes the chunking comparison measurable.
#
# The unanswerable rows are not waste either - they are free ground truth for the
# refusal guardrails, which is why nothing is filtered out here.
!python -m ragoa.data.docbuilder --source-lang hi --langs hi bn ta mr

In [ ]:
#@title 6. Build the index on the GPU
MAX_DOCS = 0  #@param {type:"integer"}
BATCH_SIZE = 256  #@param [64, 128, 256, 512] {type:"raw"}

import os
import time

# The committed default is 16, which is the ceiling on a shared-memory 8 GB Mac.
# A T4 has its own 15 GB and wants a far larger batch.
os.environ["EMBED_BATCH_SIZE"] = str(BATCH_SIZE)

limit = f"--docs {MAX_DOCS}" if MAX_DOCS else ""
t0 = time.time()
!python scripts/build_index.py --encoder st --device cuda {limit}
print(f"\nwall clock: {(time.time() - t0) / 60:.1f} min")
!du -sh data/index; ls -la data/index

In [ ]:
#@title 7. Prove the index works before downloading half a gigabyte
# Retrieval only, no LLM: this checks that vectors, offsets and the lexical index
# all line up. Queries are drawn from the corpus itself, and the gold spans tell us
# whether the retrieved chunk actually contains the answer - so this is a real
# recall check, not just "it returned something".
from pathlib import Path

import pyarrow.parquet as pq

from ragoa.factory import load_retriever
from ragoa.schemas import Deadline, Trace

# use_rerank stays on: with no ONNX export present the factory falls back to the
# torch cross-encoder, which is slow but correct, and this is a correctness check.
retriever, manifest = load_retriever(
    index_dir=Path("data/index"),
    encoder_kind="st", use_rerank=True, use_sparse=True, device="cuda",
)
print(f"index: {manifest['n_chunks']:,} chunks over {manifest['n_docs']:,} docs, "
      f"strategy={manifest['strategy']}\n")

rows = pq.read_table("data/corpus/docs.parquet",
                     columns=["doc_id", "eng_query", "eng_answer", "answerable",
                              "gold_starts", "gold_ends"]).to_pylist()
probes = [r for r in rows if r["answerable"] and r["gold_starts"]][:8]

hits = 0
for row in probes:
    trace = Trace(request_id="probe")
    results = retriever.retrieve(row["eng_query"], Deadline(2000.0), trace, top_k=3)
    gold = list(zip(row["gold_starts"], row["gold_ends"]))
    hit = any(c.chunk.doc_id == row["doc_id"] and any(c.chunk.overlaps(g) for g in gold)
              for c in results)
    hits += hit
    print(f"{'HIT ' if hit else 'MISS'} {row['eng_query'][:58]:<58} "
          f"-> {results[0].chunk.text[:70].strip()!r}")

print(f"\ngold chunk in top-3 for {hits}/{len(probes)} probes")
print("(a couple of misses is normal; a zero here means the index is broken)")

In [ ]:
#@title 8. Package the artifacts
# Corpus tables travel with the index because the evaluation scripts need the gold
# spans and the multilingual queries, and they are small next to the vectors.
import time

t0 = time.time()
!tar czf /content/ragoa-index.tar.gz data/index data/corpus
print(f"packed in {time.time() - t0:.0f}s")
!ls -lh /content/ragoa-index.tar.gz

## Getting the archive out

Pick whichever is less friction. Drive needs no accounts or tokens and is the fast path to a working demo; the Hugging Face route is the one the deployed API uses, because Fly.io needs somewhere to pull the index from at boot and cannot read your Drive.

Run **9a** or **9b**, not both.

In [ ]:
#@title 9a. Copy to Google Drive (no tokens needed)
from google.colab import drive

drive.mount("/content/drive")
!mkdir -p "/content/drive/MyDrive/ragoa"
!cp /content/ragoa-index.tar.gz "/content/drive/MyDrive/ragoa/"
!ls -lh "/content/drive/MyDrive/ragoa/"
print("\nDownload it from Drive, then on the laptop, from the repo root:")
print("  tar xzf ~/Downloads/ragoa-index.tar.gz")

In [ ]:
#@title 9b. Push to a private Hugging Face dataset repo (what Fly.io pulls at boot)
HF_REPO = "your-username/ragoa-index"  #@param {type:"string"}

from huggingface_hub import HfApi, notebook_login

notebook_login()  # needs a token with write access

api = HfApi()
api.create_repo(HF_REPO, repo_type="dataset", private=True, exist_ok=True)
api.upload_file(
    path_or_fileobj="/content/ragoa-index.tar.gz",
    path_in_repo="ragoa-index.tar.gz",
    repo_id=HF_REPO,
    repo_type="dataset",
)
print(f"\nuploaded to https://huggingface.co/datasets/{HF_REPO}")
print("Locally:  huggingface-cli download --repo-type dataset "
      f"{HF_REPO} ragoa-index.tar.gz --local-dir .")